In [ ]:
pip install ugot opencv-python

In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.10") # use YOUR IP address

import time

got.load_models(["line_recognition", "color_recognition"])
got.set_track_recognition_line(0)

got.transform_adaption_control(False)
got.transform_set_chassis_height(6)

def swipe():
    """Swipe away large cubes (unripe fruit)."""
    got.mechanical_joint_control(0, -35, -60, 500)
    time.sleep(1) # wait 1 second
    got.mechanical_joint_control(-40, -25, -40, 500)
    time.sleep(1) # wait 1 second
    got.mechanical_joint_control(40, -25, -40, 500)
    time.sleep(1) # wait 1 second
    
while True:
    color_info = got.get_color_total_info()
    # print(color_info)
    if color_info[1] == "Cube":
        got.mecanum_stop()
        swipe()
    line_info = got.get_single_track_total_info()
    if line_info:
        offset = line_info[0]
        rot = int(offset*0.5)
        # got.mecanum_move_xyz(0, 20, rot)
        if rot < 0:
            turn = 3 # right
        else:
            turn = 2 # left
        got.transform_move_turn(0, 20, turn, abs(rot))

192.168.1.10:50051


# Model training
First run the cell below to install all required packages.

In [ ]:
pip install ultralytics ugot opencv-python

## 1. Collect data
Move the UGOT / cubes around so that you get images when the cube is near/far, rotated to different positions, under different lighting conditions, partially blocked, multiple cubes in the same picture, etc. Think about what kind of conditions the robot might encounter during the competition.

In [ ]:
# pastebin.com/UsFavtqJ

# Save images from UGOT video feed at regular time intervals
from ugot import ugot
import cv2
import numpy as np
import time
import os

SAVE_DIR = "cubes" # changed folder for demo purposes
os.makedirs(SAVE_DIR, exist_ok=True)

got = ugot.UGOT()
got.initialize("192.168.1.54")
got.open_camera()
got.transform_set_chassis_height(7)

counter = 20    # with multiple runs, change this to one after the last captured image name to avoid overwriting images
interval = 3   # how many seconds between each picture

print("Auto-capturing images. Press 'q' to stop.")

last_time = time.time()

try:
    while True:
        frame = got.read_camera_data()
        if frame is not None:
            nparr = np.frombuffer(frame, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            cv2.imshow("UGOT Camera", img)

            # Auto-save
            if time.time() - last_time >= interval:
                filename = f"{SAVE_DIR}/cubes_{counter:04d}.jpg"
                cv2.imwrite(filename, img)
                print(f"Saved: {filename}")
                counter += 1
                last_time = time.time()

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

finally:
    cv2.destroyAllWindows()
    print("Done.")


192.168.1.54:50051
Auto-capturing images. Press 'q' to stop.
Saved: cubes/cubes_0001.jpg
Saved: cubes/cubes_0002.jpg
Saved: cubes/cubes_0003.jpg
Saved: cubes/cubes_0004.jpg
Saved: cubes/cubes_0005.jpg
Saved: cubes/cubes_0006.jpg
Saved: cubes/cubes_0007.jpg
Saved: cubes/cubes_0008.jpg
Saved: cubes/cubes_0009.jpg
Saved: cubes/cubes_0010.jpg
Saved: cubes/cubes_0011.jpg
Saved: cubes/cubes_0012.jpg
Saved: cubes/cubes_0013.jpg
Saved: cubes/cubes_0014.jpg
Saved: cubes/cubes_0015.jpg
Saved: cubes/cubes_0016.jpg
Saved: cubes/cubes_0017.jpg
Saved: cubes/cubes_0018.jpg
Saved: cubes/cubes_0019.jpg
Done.


## 2. Train the model
Before running the cell below, annotate the data using a tool such as roboflow, then download the dataset in the appropriate format.

In [ ]:
from ultralytics import YOLO

# Use pretrained model from YOLO
model = YOLO("yolo26n.pt")

# Train on new roboflow data
model.train(data=r"C:\Users\miche\TTA-Projects\AIMS cubes.v2-add-version.yolov11\data.yaml", epochs=50, imgsz=512)

## 3. Test the model
Change the string below to the filepath of the model.

In [1]:
# pastebin.com/YMnCXw1q
from ultralytics import YOLO

# trained = YOLO(r"C:\Users\miche\TTA-Projects\runs\detect\train\weights\best.pt")
trained = YOLO("yolo26n.pt")

from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.54")
got.open_camera()

import cv2
import numpy as np

# Helper: Draw bounding boxes
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes  # bounding boxes

        for box in boxes:
            # xyxy format: [x1, y1, x2, y2]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            # Confidence & label
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = r.names[cls_id]

            # Draw rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2)
    return frame

while True:
    try:
        frame = got.read_camera_data()
        if frame is not None:
            nparr = np.frombuffer(frame, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            # Run YOLO detection
            results = trained(img, verbose=False)

            # Draw output
            output = draw_detections(img, results)

            # Show
            cv2.imshow("YOLO Detection - AIMS", output)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        break

cv2.destroyAllWindows()

192.168.1.54:50051
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
